# Real-data example: Iris morphology across species

This notebook shows how PyStars can be used on a small, real biological dataset.
We compare continuous flower morphology measurements across three *Iris* species, using the same workflow you might apply to experimental measurements from animals, cultures, tissues, or imaging assays.

**Research question:** Do sepal and petal measurements differ between *Iris setosa*, *Iris versicolor*, and *Iris virginica*?

**Data source:** Fisher, R. A. (1936). *Iris* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56C76

The dataset is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/legalcode). Measurements are in centimeters.

In [1]:
import pandas as pd

import pystars as ps

## Load the data

The UCI Iris CSV has no header, so we supply column names manually and clean the species labels for readability.

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
column_names = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm", "species"]

df = pd.read_csv(url, header=None, names=column_names)
df["species"] = df["species"].str.removeprefix("Iris-")

df.head()

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [3]:
# Sample sizes per species
df["species"].value_counts()

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

## Inspect the measurements

Before running statistical tests, inspect group sizes and rough measurement distributions. This helps catch obvious data-loading problems and gives context for the effect sizes.

In [4]:
features = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]

summary_stats = df.groupby("species")[features].agg(
    ["count", "mean", "std", "median", "min", "max"]
)
summary_stats.round(2)

sepal_length_cm                              sepal_width_cm        \
                     count  mean   std median  min  max          count  mean   
species                                                                        
setosa                  50  5.01  0.35    5.0  4.3  5.8             50  3.42   
versicolor              50  5.94  0.52    5.9  4.9  7.0             50  2.77   
virginica               50  6.59  0.64    6.5  4.9  7.9             50  2.97   

                         ... petal_length_cm                  petal_width_cm  \
             std median  ...             std median  min  max          count   
species                  ...                                                   
setosa      0.38    3.4  ...            0.17   1.50  1.0  1.9             50   
versicolor  0.31    2.8  ...            0.47   4.35  3.0  5.1             50   
virginica   0.32    3.0  ...            0.55   5.55  4.5  6.9             50   

                                         
            mean   std median  min  max  
species                                  
setosa      0.24  0.11    0.2  0.1  0.6  
versicolor  1.33  0.20    1.3  1.0  1.8  
virginica   2.03  0.27    2.0  1.4  2.5  

[3 rows x 24 columns]

## Primary analysis: petal length

We start with a single measurement and let PyStars choose the appropriate omnibus test based on normality and equal-variance checks. Because there are three species, a significant omnibus test is followed by the matching post-hoc comparisons.

In [5]:
result = ps.test(df, value="petal_length_cm", group="species")
result.show()

╭───────────────────────────────────────────────── Welch's ANOVA ─────────────────────────────────────────────────╮
│  Field      Value                                                                                               │
│  Test       Welch's ANOVA                                                                                       │
│  Statistic  1827                                                                                                │
│  p-value    <0.0001                                                                                             │
│  np2        0.9413                                                                                              │
│  Assumption      Method   p-value  Verdict                                                                      │
│  normality       shapiro  0.05465  not rejected                                                                 │
│  equal variance  levene   <0.0001  rejected                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
# Tidy one-row export
result.to_dataframe()

,test,statistic,p_value,np2,normality_method,normality_statistic,normality_p,equal_variance_method,equal_variance_statistic,equal_variance_p,posthoc
0,Welch's ANOVA,1826.580952,2.853131e-66,0.941319,shapiro,0.954946,0.05465,levene,19.720055,2.589296e-08,Games-Howell


In [7]:
# Pairwise post-hoc comparisons selected by the dispatcher
result.posthoc.pairwise.sort_values("p").reset_index(drop=True)

,A,B,diff,p
0,versicolor,virginica,-1.292,0.000000e+00
1,setosa,virginica,-4.088,4.884981e-15
2,setosa,versicolor,-2.796,2.153833e-14


## Batch analysis across all morphology features

For exploratory work it is convenient to run the same workflow on several measurements and collect the results in one table. Because this is a batch of related tests, we correct the primary p-values with Benjamini-Hochberg FDR (`fdr_bh`).

In [8]:
feature_labels = {
    "sepal_length_cm": "Sepal length (cm)",
    "sepal_width_cm": "Sepal width (cm)",
    "petal_length_cm": "Petal length (cm)",
    "petal_width_cm": "Petal width (cm)",
}

results = []
measurements = []
for feature in features:
    results.append(ps.test(df, value=feature, group="species"))
    measurements.append(feature_labels[feature])

summary = ps.to_dataframe(results, p_adjust="fdr_bh")
summary.insert(0, "measurement", measurements)
display_cols = [
    "measurement",
    "test",
    "statistic",
    "p_value",
    "p_adjusted",
    "reject",
    "p_adjust_method",
    "np2",
    "epsilon_squared",
    "normality_p",
    "equal_variance_p",
    "posthoc",
]
summary = summary[[col for col in display_cols if col in summary.columns]]
summary = summary.sort_values("p_adjusted").reset_index(drop=True)
summary

,measurement,test,statistic,p_value,p_adjusted,reject,p_adjust_method,np2,epsilon_squared,normality_p,equal_variance_p,posthoc
0,Petal length (cm),Welch's ANOVA,1826.580952,2.853131e-66,1.141252e-65,True,fdr_bh,0.941319,NaN,0.054650,2.589296e-08,Games-Howell
1,Petal width (cm),Kruskal-Wallis test,131.093353,3.415388e-29,6.830776e-29,True,fdr_bh,NaN,0.879821,0.000002,3.301950e-08,Dunn's test
2,Sepal length (cm),Welch's ANOVA,138.908285,1.505059e-28,2.006745e-28,True,fdr_bh,0.618706,NaN,0.258315,2.258528e-03,Games-Howell
3,Sepal width (cm),One-way ANOVA,47.364461,1.327917e-16,1.327917e-16,True,fdr_bh,0.391881,NaN,0.180896,5.248270e-01,Tukey HSD


## Interpretation

All four morphology measurements differ strongly across species, even after Benjamini-Hochberg FDR correction across the four primary tests. The dispatcher selected Welch's ANOVA when normality was acceptable but equal variances were rejected, Kruskal-Wallis when normality was rejected, and one-way ANOVA when both assumptions were acceptable. For significant omnibus tests, it then ran the matching post-hoc test: Games-Howell after Welch's ANOVA, Dunn's test after Kruskal-Wallis, and Tukey HSD after one-way ANOVA.